In [1]:
import os
import time
import requests
import pandas as pd
from tqdm import tqdm
from bibtexparser.bwriter import BibTexWriter
from bibtexparser.bibdatabase import BibDatabase
import bibtexparser

# --- 1. CONFIGURATION ---
INPUT_CSV_FILE = 'beetle_citations.csv'
OUTPUT_DIR_PDFS = 'Downloaded_Open_Access_PDFs'
# --- NEW: A single CSV file will now be the primary log ---
PROCESSING_LOG_CSV = 'processing_log.csv'
# These are now final outputs, not logs
OUTPUT_BIB_DOWNLOADED = 'open_access_citations.bib'
OUTPUT_BIB_PAYWALLED = 'paywalled_citations.bib'
YOUR_EMAIL = "gmarais@ufl.edu"

# --- 2. HELPER FUNCTIONS ---

def get_oa_pdf_url(doi, email):
    """Queries the Unpaywall API to find a legal Open Access PDF URL for a given DOI."""
    try:
        url = f"https://api.unpaywall.org/v2/{doi}?email={email}"
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        data = response.json()
        if data.get('best_oa_location') and data['best_oa_location'].get('url_for_pdf'):
            return data['best_oa_location']['url_for_pdf']
    except requests.exceptions.RequestException:
        pass
    return None

def sanitize_filename(name):
    """Removes illegal characters from a string to make it a valid filename."""
    if not isinstance(name, str):
        name = "No Title"
    illegal_chars = ['<', '>', ':', '"', '/', '\\', '|', '?', '*', '\n', '\r', '\t']
    for char in illegal_chars:
        name = name.replace(char, '_')
    return name[:150]

def download_pdf(pdf_url, filepath):
    """Downloads a PDF from a URL and saves it to a given path."""
    try:
        headers = {'User-Agent': 'Mozilla/5.0'}
        response = requests.get(pdf_url, headers=headers, stream=True, timeout=30)
        response.raise_for_status()
        with open(filepath, 'wb') as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
        return True
    except requests.exceptions.RequestException:
        return False

# --- NEW: Function to append a dictionary to our CSV log ---
def append_to_log_csv(log_entry, filename):
    """Appends a new row to the CSV log file."""
    # Create a DataFrame from the single log entry dictionary
    df_to_append = pd.DataFrame([log_entry])
    # Check if the file exists to determine if we need to write the header
    write_header = not os.path.exists(filename)
    # Append to the CSV file
    df_to_append.to_csv(filename, mode='a', header=write_header, index=False)

# --- 3. MAIN SCRIPT ---

if __name__ == "__main__":
    print("Starting Open Access PDF downloader script...")
    os.makedirs(OUTPUT_DIR_PDFS, exist_ok=True)
    
    # --- MODIFIED: Resume logic now based entirely on the CSV log ---
    processed_dois = set()
    if os.path.exists(PROCESSING_LOG_CSV):
        print(f"Loading previously processed items from '{PROCESSING_LOG_CSV}'...")
        log_df = pd.read_csv(PROCESSING_LOG_CSV)
        # Ensure the 'doi' column is treated as string
        processed_dois = set(log_df['doi'].astype(str))
        print(f"Found {len(processed_dois)} DOIs already processed. They will be skipped.")

    try:
        df = pd.read_csv(INPUT_CSV_FILE)
        df.dropna(subset=['doi'], inplace=True)
        # Ensure 'doi' column is string for comparison
        df['doi'] = df['doi'].astype(str)
        df_to_process = df[~df['doi'].isin(processed_dois)].copy()
        print(f"Found {len(df_to_process)} new papers to process.")
    except FileNotFoundError:
        print(f"ERROR: Input file '{INPUT_CSV_FILE}' not found.")
        exit()

    pbar = tqdm(df_to_process.iterrows(), total=len(df_to_process), desc="Processing Papers")
    
    for index, row in pbar:
        doi = row['doi']
        clean_doi = doi.split('doi.org/')[-1] # For API calls
        
        pdf_url = get_oa_pdf_url(clean_doi, YOUR_EMAIL)
        
        status = ''
        
        if pdf_url:
            filename = sanitize_filename(row.get('title', 'No Title')) + '.pdf'
            filepath = os.path.join(OUTPUT_DIR_PDFS, filename)
            
            if download_pdf(pdf_url, filepath):
                status = 'downloaded'
            else:
                status = 'download_failed' # Could be treated as paywalled
        else:
            status = 'paywalled'
            
        # Prepare the log entry and append it to the CSV
        log_entry = {
            'doi': doi,
            'status': status,
            'title': row.get('title', ''),
            'authors': row.get('authors', ''),
            'year': row.get('year', ''),
            'journal': row.get('journal', '')
        }
        append_to_log_csv(log_entry, PROCESSING_LOG_CSV)
        time.sleep(0.1)

    print("\n--- Processing Complete! ---")
    print("All progress saved to 'processing_log.csv'.")
    
    # --- NEW: Generate final BibTeX files from the completed CSV log ---
    print("Generating final BibTeX files...")
    
    if not os.path.exists(PROCESSING_LOG_CSV):
        print("Log file not found. Nothing to write to BibTeX.")
    else:
        final_log_df = pd.read_csv(PROCESSING_LOG_CSV)
        
        # Separate into downloaded and paywalled
        downloaded_df = final_log_df[final_log_df['status'] == 'downloaded']
        paywalled_df = final_log_df[final_log_df['status'] != 'downloaded'] # Includes paywalled and failed

        # Helper to convert a DataFrame to a BibDatabase object
        def df_to_bib_database(df):
            db = BibDatabase()
            for index, row in df.iterrows():
                authors_str = str(row.get('authors', ''))
                year_str = str(int(row['year'])) if pd.notna(row.get('year')) else "ND"
                first_author_full = authors_str.split(';')[0].strip() if authors_str else "Unknown"
                first_author_lastname = first_author_full.split(',')[0].strip()
                clean_doi_key = str(row.get('doi', '')).split('doi.org/')[-1].replace('/', '-')
                citation_key = f"{first_author_lastname}{year_str}_{clean_doi_key[:25]}"

                entry = {
                    'ENTRYTYPE': 'article', 'ID': citation_key,
                    'author': authors_str.replace(';', ' and '),
                    'title': str(row.get('title', 'No Title')), 'year': year_str,
                    'journal': str(row.get('journal', '')), 'doi': str(row.get('doi', ''))
                }
                db.entries.append(entry)
            return db

        # Create and write the downloaded BibTeX file
        downloaded_db = df_to_bib_database(downloaded_df)
        writer = BibTexWriter()
        writer.indent = '    '
        with open(OUTPUT_BIB_DOWNLOADED, 'w', encoding='utf-8') as bibfile:
            bibfile.write(writer.write(downloaded_db))
        print(f"✅ Created '{OUTPUT_BIB_DOWNLOADED}' with {len(downloaded_db.entries)} entries.")
        
        # Create and write the paywalled BibTeX file
        paywalled_db = df_to_bib_database(paywalled_df)
        with open(OUTPUT_BIB_PAYWALLED, 'w', encoding='utf-8') as bibfile:
            bibfile.write(writer.write(paywalled_db))
        print(f"🚫 Created '{OUTPUT_BIB_PAYWALLED}' with {len(paywalled_db.entries)} entries.")

Starting Open Access PDF downloader script...
Loading previously processed items from 'processing_log.csv'...
Found 135991 DOIs already processed. They will be skipped.
Found 22503 new papers to process.


Processing Papers: 100%|██████████| 22503/22503 [5:43:59<00:00,  1.09it/s]   



--- Processing Complete! ---
All progress saved to 'processing_log.csv'.
Generating final BibTeX files...
✅ Created 'open_access_citations.bib' with 20126 entries.
🚫 Created 'paywalled_citations.bib' with 138368 entries.
